# 🔍 Hybrid Search with Pinecone + LangChain

**Architecture:** Dense (Semantic) + Sparse (BM25 Keyword) vectors combined in a single Pinecone index using `dotproduct` metric.

| Component | Tool |
|---|---|
| Dense Embeddings | `all-MiniLM-L6-v2` (HuggingFace) |
| Sparse Encoding | BM25 (`pinecone-text`) |
| Vector DB | Pinecone Serverless |
| Retriever | `PineconeHybridSearchRetriever` (LangChain) |

> **What is Hybrid Search?**  
> Traditional keyword search (BM25) is great for exact matches. Semantic/dense search handles meaning and context. Hybrid search merges both signals, significantly improving retrieval quality for RAG pipelines.

## 1. Install Dependencies

In [1]:
# Install all required packages
!pip install --quiet --upgrade \
    pinecone \
    pinecone-text \
    langchain \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    sentence-transformers \
    python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load Environment Variables

All secrets are stored in a `.env` file — **never hardcode API keys in notebooks.**

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # Loads from .env in the current directory

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
HF_TOKEN        = os.getenv("HF_TOKEN")

# Inject HF token so HuggingFace Hub can authenticate
os.environ["HF_TOKEN"] = HF_TOKEN

assert PINECONE_API_KEY, "❌ PINECONE_API_KEY missing from .env"
assert HF_TOKEN,         "❌ HF_TOKEN missing from .env"
print("✅ Environment variables loaded successfully.")

✅ Environment variables loaded successfully.


## 3. Initialize Pinecone Client & Create Index

We use **Pinecone Serverless** (AWS us-east-1).  
Hybrid search requires the `dotproduct` metric — it's the only metric that supports sparse+dense vectors together.

In [4]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME  = "hybrid-search-langchain-pinecone"
DIMENSION   = 384   # all-MiniLM-L6-v2 output dimension
METRIC      = "dotproduct"   # Required for hybrid (sparse + dense)
CLOUD       = "aws"
REGION      = "us-east-1"

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index only if it doesn't already exist
existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creating index: '{INDEX_NAME}' ...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(cloud=CLOUD, region=REGION),
    )
    print("✅ Index created.")
else:
    print(f"✅ Index '{INDEX_NAME}' already exists — skipping creation.")

index = pc.Index(INDEX_NAME)
print(f"\nIndex stats: {index.describe_index_stats()}")

Creating index: 'hybrid-search-langchain-pinecone' ...
✅ Index created.

Index stats: {'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '154',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 11 Mar 2026 14:47:38 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '39',
                                    'x-pinecone-request-latency-ms': '38',
                                    'x-pinecone-response-duration-ms': '40'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


## 4. Dense Embeddings — HuggingFace (`all-MiniLM-L6-v2`)

`all-MiniLM-L6-v2` is a lightweight 384-dim model that's fast and accurate for sentence similarity tasks.  
It powers the **semantic (dense)** side of our hybrid retriever.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},    # Change to "cuda" if GPU is available
    encode_kwargs={"normalize_embeddings": True},  # Normalize for dotproduct
)

# Quick sanity check
test_vec = embeddings.embed_query("Hello world")
print(f"✅ Embedding model loaded. Vector dimension: {len(test_vec)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded. Vector dimension: 384


## 5. Sparse Encoding — BM25

BM25 (Best Match 25) is a classic TF-IDF-based ranking function used in search engines.  
Here we **fit** it on our corpus so it learns token frequencies, then **dump** the state to JSON for reproducibility.

In [6]:
from pinecone_text.sparse import BM25Encoder

# Sample corpus — replace or extend with your own documents
corpus = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]

# Fit BM25 on the corpus (learns IDF statistics)
bm25_encoder = BM25Encoder()
bm25_encoder.fit(corpus)

# Persist fitted parameters to disk for future reuse
bm25_encoder.dump("bm25_values.json")
print("✅ BM25 encoder fitted and saved to bm25_values.json")

  0%|          | 0/3 [00:00<?, ?it/s]

✅ BM25 encoder fitted and saved to bm25_values.json


In [7]:
# Load from saved file (demonstrates reproducible reloading)
bm25_encoder = BM25Encoder().load("bm25_values.json")
print("✅ BM25 encoder loaded from bm25_values.json")

# Inspect a sample sparse encoding
sample_sparse = bm25_encoder.encode_documents(["I visited Paris"])
print(f"\nSample sparse vector (first doc):\n  indices: {sample_sparse[0]['indices'][:5]}\n  values:  {[round(v,4) for v in sample_sparse[0]['values'][:5]]}")

✅ BM25 encoder loaded from bm25_values.json

Sample sparse vector (first doc):
  indices: [2395889254, 4239951674]
  values:  [0.5584, 0.5584]


## 6. Build the Hybrid Retriever

In [8]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index,
    top_k=3,           # Number of results to retrieve
    alpha=0.5,         # 0 = pure sparse (BM25), 1 = pure dense (semantic), 0.5 = balanced
)

print("✅ PineconeHybridSearchRetriever initialized.")
print(f"   alpha={retriever.alpha} (balanced hybrid mode)")

✅ PineconeHybridSearchRetriever initialized.
   alpha=0.5 (balanced hybrid mode)


## 7. Index Documents

In [9]:
# Add the corpus documents into Pinecone (upserts dense + sparse vectors together)
retriever.add_texts(corpus)
print(f"✅ {len(corpus)} documents indexed into Pinecone.")

# Verify index count
import time
time.sleep(2)  # Allow Pinecone a moment to reflect the upsert
stats = index.describe_index_stats()
print(f"   Total vectors in index: {stats['total_vector_count']}")

  0%|          | 0/1 [00:00<?, ?it/s]

✅ 3 documents indexed into Pinecone.
   Total vectors in index: 3


## 8. Run Hybrid Queries

In [10]:
def run_query(query: str):
    """Run a hybrid search query and display results."""
    print(f"\n🔎 Query: \"{query}\"")
    print("-" * 50)
    results = retriever.invoke(query)
    for i, doc in enumerate(results, 1):
        print(f"  [{i}] {doc.page_content}")
    return results

# --- Test Queries ---
run_query("What city did I visit first?")
run_query("Which trip was most recent?")
run_query("New York")


🔎 Query: "What city did I visit first?"
--------------------------------------------------
  [1] In 2023, I visited Paris
  [2] In 2022, I visited New York
  [3] In 2021, I visited New Orleans

🔎 Query: "Which trip was most recent?"
--------------------------------------------------
  [1] In 2023, I visited Paris
  [2] In 2022, I visited New York
  [3] In 2021, I visited New Orleans

🔎 Query: "New York"
--------------------------------------------------
  [1] In 2023, I visited Paris
  [2] In 2022, I visited New York
  [3] In 2021, I visited New Orleans


[Document(metadata={'score': 0.0906024}, page_content='In 2023, I visited Paris'),
 Document(metadata={'score': 0.491866261}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.240956515}, page_content='In 2021, I visited New Orleans')]

## 9. Alpha Tuning — Keyword vs Semantic Balance

The `alpha` parameter controls the blend between sparse and dense scores.  
Experiment below to understand how retrieval changes.

In [11]:
query = "city visited in 2022"

for alpha_val in [0.0, 0.5, 1.0]:
    label = {0.0: "Pure BM25 (keyword)", 0.5: "Hybrid (balanced)", 1.0: "Pure Dense (semantic)"}[alpha_val]
    retriever.alpha = alpha_val
    results = retriever.invoke(query)
    print(f"\nalpha={alpha_val} | {label}")
    for i, doc in enumerate(results, 1):
        print(f"  [{i}] {doc.page_content}")

# Reset to default
retriever.alpha = 0.5


alpha=0.0 | Pure BM25 (keyword)
  [1] In 2023, I visited Paris
  [2] In 2022, I visited New York
  [3] In 2021, I visited New Orleans

alpha=0.5 | Hybrid (balanced)
  [1] In 2023, I visited Paris
  [2] In 2022, I visited New York
  [3] In 2021, I visited New Orleans

alpha=1.0 | Pure Dense (semantic)
  [1] In 2023, I visited Paris
  [2] In 2022, I visited New York
  [3] In 2021, I visited New Orleans


## 10. (Optional) Extend with Your Own Documents

Drop in any list of strings — the workflow is identical.

In [12]:
# ---- Extend corpus example ----
new_docs = [
    "Machine learning is transforming the healthcare industry.",
    "Large language models can generate human-like text at scale.",
    "Retrieval Augmented Generation improves LLM factual accuracy.",
]

# Re-fit BM25 on expanded corpus
all_docs = corpus + new_docs
bm25_encoder = BM25Encoder()
bm25_encoder.fit(all_docs)
bm25_encoder.dump("bm25_values.json")

# Re-initialise retriever with updated encoder
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index,
    top_k=3,
    alpha=0.5,
)

retriever.add_texts(new_docs)
print("✅ New documents added.")
print("Uncomment the block above to extend the index with custom documents.")

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

✅ New documents added.
Uncomment the block above to extend the index with custom documents.


## 11. Cleanup — Delete Index (Optional)

⚠️ **Only run this if you want to delete the Pinecone index and its data.**

In [ ]:
# DANGER: Deletes the entire Pinecone index
# Uncomment to run:

# pc.delete_index(INDEX_NAME)
# print(f"🗑️ Index '{INDEX_NAME}' deleted.")
print("Cleanup cell ready — uncomment to delete index.")